Chunking Practical: Most Useful Methods
This notebook covers the following methods:

RecursiveCharacterTextSplitter
TokenTextSplitter
CharacterTextSplitter
Fixed-size chunking
Content-aware chunking
Document structure-based chunking
Semantic chunking
Learning objectives
Explain chunk size and chunk overlap
Compare fixed, character, recursive, token, structural, and semantic splitting
Understand why different strategies produce different boundaries
Choose a suitable starting method for a given document type
Evaluate chunks instead of blindly selecting one splitter

Fixed-size character chunking → CharacterTextSplitter
Fixed-size token chunking → TokenTextSplitter
Recursive structure-aware chunking → RecursiveCharacterTextSplitter
Document structure-based chunking → MarkdownHeaderTextSplitter, HTML splitters, etc.
Semantic chunking → Embedding-based topic boundaries


CharacterTextSplitter
→ Primarily separator-based
→ Length measured using characters
TokenTextSplitter
→ Directly splits according to tokens
RecursiveCharacterTextSplitter
→ Tries to preserve paragraphs, lines and words
→ Not purely blind fixed-size chunking


In [2]:
from __future__ import annotations
import math
import re
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

e:\workspace\myenv10pm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Sample Document
The sample document intentionally contains multiple topics and Markdown headings.
This makes it useful for comparing basic, structure-aware, and semantic chunking.

In [3]:
document = """
# Artificial Intelligence

Artificial intelligence enables machines to perform tasks that normally require human intelligence.
AI systems can analyze information, identify patterns, and support decision-making.

## Machine Learning

Machine learning is a branch of artificial intelligence.
It enables systems to learn patterns from historical data instead of relying only on explicitly programmed rules.
Supervised learning uses labelled examples, while unsupervised learning discovers patterns in unlabelled data.

## Deep Learning

Deep learning uses neural networks containing multiple layers.
It is commonly applied to image recognition, speech processing, and natural-language applications.

# Employee Leave Policy

Employees receive twenty paid leave days during a calendar year.
Leave requests should be submitted to the reporting manager before the planned absence.
Emergency leave may be submitted later when advance notice is not possible.

## Approval Process

The manager reviews the leave balance, project schedule, and team availability.
Approved leave is recorded in the human-resource management system.

# Information Security

Employees must not share passwords or authentication codes.
Sensitive company information should be stored only in approved systems.
Suspected security incidents must be reported to the security team immediately.
""".strip()

In [4]:
print(document)

# Artificial Intelligence

Artificial intelligence enables machines to perform tasks that normally require human intelligence.
AI systems can analyze information, identify patterns, and support decision-making.

## Machine Learning

Machine learning is a branch of artificial intelligence.
It enables systems to learn patterns from historical data instead of relying only on explicitly programmed rules.
Supervised learning uses labelled examples, while unsupervised learning discovers patterns in unlabelled data.

## Deep Learning

Deep learning uses neural networks containing multiple layers.
It is commonly applied to image recognition, speech processing, and natural-language applications.

# Employee Leave Policy

Employees receive twenty paid leave days during a calendar year.
Leave requests should be submitted to the reporting manager before the planned absence.
Emergency leave may be submitted later when advance notice is not possible.

## Approval Process

The manager reviews the lea

In [5]:
from pypdf import PdfReader
reader = PdfReader("E:\workspace\RAG\Data\GAUTAM Tankala-ML.pdf")
page_texts = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        page_texts.append(text)
    
    print(
        f"Page {page_number}: "
        f"{len(text)} characters extracted"
    )
    

Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 48 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 57 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong pointing object 63 0 (offset 0)
Ignoring wrong pointing object 64 0 (offset 0)
Ignoring wrong pointing object 65 0 (offset 0)
Ignoring wrong pointing object 66 0 (offset 0)
Ignoring wrong pointing object 67 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)
Ignoring wrong pointing object 69 0 (offset 0)
Ignoring wron

Page 1: 4084 characters extracted
Page 2: 3707 characters extracted
Page 3: 4007 characters extracted


In [6]:
document = "\n\n".join(page_texts)

In [7]:
print(document)

SREELAKSHMI 
Email: sreebotintel@gmail.com 
Ph: +91 8123118604 
TECH LEAD (DATA SCIENTIST/MACHINE LEARNING ENGINEER) 
GAUTAM T 
Email: gautamvenki1111@gmail.com 
Ph: +91 7795297775 
DATA SCIENTIST & GEN AI 
 
 
PROFESSIONAL SUMMARY: 
 
● 9 years of IT industry experience as a Data Scientist/Machine Learning Engg specialized in implementing 
advanced Machine Learning, Deep Learning and Natural Language Processing algorithms upon data from 
diverse domains. Experience in building highly effici ent models to derive actionable insights for business 
environments leveraging exploratory data analysis, feature engineering, statistical modeling and predictive 
analytics. 
● Experiences in machine learning, data mining, structured and unstructured data analysis, and image data 
analysis, including feature extraction, pattern recognition, algorithm development, text mining, computer 
simulation, data modeling, databases design, model evaluation and deployment. 
● Experience with Statistical Anal

In [8]:
# 2. Helper Functions

# These helpers print chunks consistently and collect simple statistics.
from typing import Any
def print_chunks(
    title: str,
    chunks: list[Any],
    preview_limit: int = 500,
) -> None:
    print(f"\n{'=' * 90}")
    print(title)
    print(f"Total chunks: {len(chunks)}")
    print("=" * 90)
    
    for index, chunk in enumerate(chunks, start=1):

        # Get text from a LangChain Document or normal Python object
        text = getattr(
            chunk,
            "page_content",
            str(chunk),
        )

        # Get metadata if available
        metadata = getattr(
            chunk,
            "metadata",
            {},
        )

        print(
            f"\nChunk {index} | "
            f"characters={len(text)}"
        )

        if metadata:
            print("Metadata:", metadata)

        if len(text) > preview_limit:
            print(text[:preview_limit] + "...")
        else:
            print(text)

In [9]:
from typing import Any
import numpy as np


def chunk_stats(
    method: str,
    chunks: list[Any],
) -> dict[str, Any]:

    lengths = [
        len(
            getattr(
                chunk,
                "page_content",
                str(chunk),
            )
        )
        for chunk in chunks
    ]

    return {
        "method": method,
        "number_of_chunks": len(chunks),
        "minimum_length": min(lengths) if lengths else 0,
        "maximum_length": max(lengths) if lengths else 0,
        "average_length": (
            round(float(np.mean(lengths)), 2)
            if lengths
            else 0
        ),
        "total_stored_characters": sum(lengths),
    }

3. Fixed-Size Chunking
Fixed-size chunking divides content after a specified number of characters or tokens.

It is simple and predictable, but it may break:

Words
Sentences
Paragraphs

In [10]:
def fixed_size_chunking(text: str, chunk_size: int, chunk_overlap: int = 0) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")
    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must satisfy 0 <= chunk_overlap < chunk_size."
        )
    step = chunk_size - chunk_overlap
    return [
        text[start:start + chunk_size]
        for start in range(0, len(text), step)
    ]


In [11]:
text = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [12]:
len(text)

26

In [13]:
fixed_size_chunking(text, chunk_size=10, chunk_overlap=2)

['ABCDEFGHIJ', 'IJKLMNOPQR', 'QRSTUVWXYZ', 'YZ']

10 - 2 = 8

So chunks begin at positions 0, 8, 16, 24.

In [14]:
document

"SREELAKSHMI \nEmail: sreebotintel@gmail.com \nPh: +91 8123118604 \nTECH LEAD (DATA SCIENTIST/MACHINE LEARNING ENGINEER) \nGAUTAM T \nEmail: gautamvenki1111@gmail.com \nPh: +91 7795297775 \nDATA SCIENTIST & GEN AI \n \n \nPROFESSIONAL SUMMARY: \n \n● 9 years of IT industry experience as a Data Scientist/Machine Learning Engg specialized in implementing \nadvanced Machine Learning, Deep Learning and Natural Language Processing algorithms upon data from \ndiverse domains. Experience in building highly effici ent models to derive actionable insights for business \nenvironments leveraging exploratory data analysis, feature engineering, statistical modeling and predictive \nanalytics. \n● Experiences in machine learning, data mining, structured and unstructured data analysis, and image data \nanalysis, including feature extraction, pattern recognition, algorithm development, text mining, computer \nsimulation, data modeling, databases design, model evaluation and deployment. \n● Experience 

In [15]:
fixed_chunks = fixed_size_chunking(text=document, chunk_size=200, chunk_overlap=10)

In [16]:
fixed_chunks

['SREELAKSHMI \nEmail: sreebotintel@gmail.com \nPh: +91 8123118604 \nTECH LEAD (DATA SCIENTIST/MACHINE LEARNING ENGINEER) \nGAUTAM T \nEmail: gautamvenki1111@gmail.com \nPh: +91 7795297775 \nDATA SCIENTIST & G',
 'ENTIST & GEN AI \n \n \nPROFESSIONAL SUMMARY: \n \n● 9 years of IT industry experience as a Data Scientist/Machine Learning Engg specialized in implementing \nadvanced Machine Learning, Deep Learning and Na',
 'ing and Natural Language Processing algorithms upon data from \ndiverse domains. Experience in building highly effici ent models to derive actionable insights for business \nenvironments leveraging expl',
 'aging exploratory data analysis, feature engineering, statistical modeling and predictive \nanalytics. \n● Experiences in machine learning, data mining, structured and unstructured data analysis, and im',
 'is, and image data \nanalysis, including feature extraction, pattern recognition, algorithm development, text mining, computer \nsimulation, data modeling, datab

In [17]:
len(fixed_chunks)

63

In [18]:
print_chunks("Manual Fixed-Size Chunking", fixed_chunks)


Manual Fixed-Size Chunking
Total chunks: 63

Chunk 1 | characters=200
SREELAKSHMI 
Email: sreebotintel@gmail.com 
Ph: +91 8123118604 
TECH LEAD (DATA SCIENTIST/MACHINE LEARNING ENGINEER) 
GAUTAM T 
Email: gautamvenki1111@gmail.com 
Ph: +91 7795297775 
DATA SCIENTIST & G

Chunk 2 | characters=200
ENTIST & GEN AI 
 
 
PROFESSIONAL SUMMARY: 
 
● 9 years of IT industry experience as a Data Scientist/Machine Learning Engg specialized in implementing 
advanced Machine Learning, Deep Learning and Na

Chunk 3 | characters=200
ing and Natural Language Processing algorithms upon data from 
diverse domains. Experience in building highly effici ent models to derive actionable insights for business 
environments leveraging expl

Chunk 4 | characters=200
aging exploratory data analysis, feature engineering, statistical modeling and predictive 
analytics. 
● Experiences in machine learning, data mining, structured and unstructured data analysis, and im

Chunk 5 | characters=200
is, and image data 
a

What to observe
    Every chunk targets the same maximum character size.
    Fifty characters are repeated between neighbouring chunks.
    A boundary may occur in the middle of a sentence or heading.
    Overlap protects some boundary information but also creates duplication

4. CharacterTextSplitter
CharacterTextSplitter splits using one selected separator.

In this example, the separator is a blank line ("\n\n"), so it tries to keep paragraphs together.

This is easy to understand, but one separator may not be sufficient for every document.

In [19]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

In [20]:
character_splitter=CharacterTextSplitter(chunk_size=10, chunk_overlap=5, separator="\n\n",length_function=len)

In [21]:
character_chunks=character_splitter.split_text(document)

Created a chunk of size 4084, which is longer than the specified 10
Created a chunk of size 3707, which is longer than the specified 10


In [22]:
print_chunks("CharacterTextSplitter", character_chunks)


CharacterTextSplitter
Total chunks: 3

Chunk 1 | characters=4083
SREELAKSHMI 
Email: sreebotintel@gmail.com 
Ph: +91 8123118604 
TECH LEAD (DATA SCIENTIST/MACHINE LEARNING ENGINEER) 
GAUTAM T 
Email: gautamvenki1111@gmail.com 
Ph: +91 7795297775 
DATA SCIENTIST & GEN AI 
 
 
PROFESSIONAL SUMMARY: 
 
● 9 years of IT industry experience as a Data Scientist/Machine Learning Engg specialized in implementing 
advanced Machine Learning, Deep Learning and Natural Language Processing algorithms upon data from 
diverse domains. Experience in building highly effici ent...

Chunk 2 | characters=3706
Machine Learning Algorithms: 
Naïve Bayes, Decision Trees, Random Forest, K-Means Clustering, Regression Models 
Databases: 
SQL, MongoDB, Vector Databases, ChromaDB 
Cloud Technologies: 
Amazon Web Services (AWS), Microsoft Azure, Google Cloud Platform (GCP) 
Operating Systems: 
Microsoft Windows, Linux (Ubuntu) 
Data Science & Statistical Modeling: 
Generalized Linear Models (GLM), Logistic Regress

Important observation
CharacterTextSplitter is separator-driven. If an individual separator-delimited block is already larger than the target size, the output may not behave like strict fixed-width slicing.

Use it when the source has a predictable separator and you want a simple baseline.

5. RecursiveCharacterTextSplitter
This splitter tries separators recursively.

Typical order:

Paragraph → line → sentence-like boundary → word → character
It first tries to preserve large natural units. If a unit is too large, it moves to a finer separator.

In [23]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    add_start_index=True,
)

In [24]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=5,
    chunk_overlap=2,
    length_function=len,
    add_start_index=True,
)

In [25]:
text = """Artificial Intelligence

AI enables machines to perform intelligent tasks.
Machine learning is a part of AI.

Employee Leave Policy

Employees receive 20 paid leave days.
Leave must be approved by the manager.
"""

In [26]:
recursive_splitter.create_documents(
    texts=[text],
    metadatas=[{"source": "classroom_sample.md"}],
)

[Document(metadata={'source': 'classroom_sample.md', 'start_index': 0}, page_content='Artif'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 3}, page_content='ifici'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 6}, page_content='cial'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 11}, page_content='Inte'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 13}, page_content='telli'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 16}, page_content='ligen'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 19}, page_content='ence'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 25}, page_content='AI'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 28}, page_content='enab'),
 Document(metadata={'source': 'classroom_sample.md', 'start_index': 30}, page_content='ables'),
 Document(metadata={'source': 'classroom_sample.md

In [28]:
recursive_documents = recursive_splitter.create_documents(
    texts=[document],
    metadatas=[{"source": "classroom_sample.md"}],
)

In [29]:
print_chunks(RecursiveCharacterTextSplitter, recursive_documents)


<class 'langchain_text_splitters.character.RecursiveCharacterTextSplitter'>
Total chunks: 3441

Chunk 1 | characters=5
Metadata: {'source': 'classroom_sample.md', 'start_index': 0}
SREEL

Chunk 2 | characters=5
Metadata: {'source': 'classroom_sample.md', 'start_index': 3}
ELAKS

Chunk 3 | characters=5
Metadata: {'source': 'classroom_sample.md', 'start_index': 6}
KSHMI

Chunk 4 | characters=4
Metadata: {'source': 'classroom_sample.md', 'start_index': 13}
Emai

Chunk 5 | characters=4
Metadata: {'source': 'classroom_sample.md', 'start_index': 15}
ail:

Chunk 6 | characters=4
Metadata: {'source': 'classroom_sample.md', 'start_index': 20}
sree

Chunk 7 | characters=5
Metadata: {'source': 'classroom_sample.md', 'start_index': 22}
eebot

Chunk 8 | characters=5
Metadata: {'source': 'classroom_sample.md', 'start_index': 25}
otint

Chunk 9 | characters=5
Metadata: {'source': 'classroom_sample.md', 'start_index': 28}
ntel@

Chunk 10 | characters=5
Metadata: {'source': 'classroom_sample.md', 'sta

Why it is commonly used
It works well for general text
It tries to preserve natural boundaries
It still controls chunk size
It is a strong baseline for PDFs, reports, policies, and articles

6. TokenTextSplitter
LLMs and embedding models process tokens rather than raw characters.

Token-based chunking is useful when:

The embedding model has a strict token limit
You need predictable model-input sizing
Character counts do not accurately represent model usage
This example uses the cl100k_base tokenizer.

In [30]:
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=50,
    chunk_overlap=15,
)

In [31]:
token_chunks = token_splitter.split_text(document)

In [35]:
print_chunks(TokenTextSplitter,token_chunks)


<class 'langchain_text_splitters.base.TokenTextSplitter'>
Total chunks: 69

Chunk 1 | characters=134
SREELAKSHMI 
Email: sreebotintel@gmail.com 
Ph: +91 8123118604 
TECH LEAD (DATA SCIENTIST/MACHINE LEARNING ENGINEER) 
GAUTAM T 
Email:

Chunk 2 | characters=143
/MACHINE LEARNING ENGINEER) 
GAUTAM T 
Email: gautamvenki1111@gmail.com 
Ph: +91 7795297775 
DATA SCIENTIST & GEN AI 
 
 
PROFESSIONAL SUMMARY:

Chunk 3 | characters=254
DATA SCIENTIST & GEN AI 
 
 
PROFESSIONAL SUMMARY: 
 
● 9 years of IT industry experience as a Data Scientist/Machine Learning Engg specialized in implementing 
advanced Machine Learning, Deep Learning and Natural Language Processing algorithms upon data

Chunk 4 | characters=320
 implementing 
advanced Machine Learning, Deep Learning and Natural Language Processing algorithms upon data from 
diverse domains. Experience in building highly effici ent models to derive actionable insights for business 
environments leveraging exploratory data analysis, feature eng